In [ ]:
!pip install opencv-python

In [ ]:
!pip install -U ultralytics

In [ ]:
# ================================================================
# BƯỚC 1: TẠO ẢNH LOW-LIGHT TỔNG HỢP (theo phương pháp 3L-YOLO)
# Tăng cường dữ liệu thiếu sáng để model học nhận diện ban đêm
# ================================================================
import cv2
import numpy as np
import shutil
from pathlib import Path

def synthesize_low_light(image, brightness_range=(0.6, 0.8),
                         gamma_range=(2.0, 5.0),
                         noise_std_range=(0.1, 0.3)):
    """
    Tổng hợp ảnh thiếu sáng từ ảnh gốc theo phương pháp 3L-YOLO.
    4 bước: Đánh giá → Giảm sáng → Gamma → Thêm nhiễu.
    """
    img = image.astype(np.float32) / 255.0

    # Bước 1: Chỉ xử lý ảnh đủ sáng
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    if gray.mean() < 80:
        return image

    # Bước 2: Giảm sáng ngẫu nhiên xuống 60-80%
    brightness_factor = np.random.uniform(*brightness_range)
    img = img * brightness_factor

    # Bước 3: Gamma correction mô phỏng thiếu sáng
    gamma = np.random.uniform(*gamma_range)
    img = np.power(np.clip(img, 0, 1), gamma)

    # Bước 4: Thêm nhiễu Gaussian + Poisson
    noise_std = np.random.uniform(*noise_std_range)
    gaussian_noise = np.random.normal(0, noise_std, img.shape).astype(np.float32)
    img = img + gaussian_noise

    img_poisson = np.clip(img * 255, 0, 255).astype(np.uint8)
    img_poisson = np.random.poisson(img_poisson.astype(np.float32) / 10.0) * 10.0
    img = np.clip(img_poisson, 0, 255).astype(np.uint8)

    return img


def augment_dataset_low_light(src_img_dir, ratio=0.5):
    """
    Tạo ảnh low-light và lưu cùng thư mục gốc (cùng labels).
    ratio: tỷ lệ ảnh được tạo bản low-light.
    """
    src_path = Path(src_img_dir)
    label_dir = src_path.parent / "labels"

    images = list(src_path.glob("*.jpg")) + list(src_path.glob("*.png"))
    selected = np.random.choice(images, size=int(len(images) * ratio), replace=False)

    count = 0
    for img_path in selected:
        img = cv2.imread(str(img_path))
        if img is None:
            continue
        dark_img = synthesize_low_light(img)
        save_name = f"lowlight_{img_path.name}"
        cv2.imwrite(str(src_path / save_name), dark_img)

        # Copy label file tương ứng (giữ nguyên annotation)
        label_path = label_dir / img_path.with_suffix(".txt").name
        if label_path.exists():
            shutil.copy(label_path, label_dir / f"lowlight_{label_path.name}")
        count += 1

    print(f"✅ Đã tạo {count} ảnh low-light tổng hợp trong {src_path}")

In [ ]:
import os
from pathlib import Path
from ultralytics import YOLO

KAGGLE_INPUT = Path("/kaggle/input")
WORKING_DIR = Path("/kaggle/working")

# 1. Tìm file model .pt
MODEL_PATH = next(KAGGLE_INPUT.rglob("*.pt"), None)
if not MODEL_PATH:
    raise FileNotFoundError("Không tìm thấy model .pt nào được đính kèm!")
print(f"✅ Đã tìm thấy model: {MODEL_PATH}")

# 2. Tự động tìm thư mục dataset (chứa thư mục 'train/images')
DATASET_PATH = None
for p in KAGGLE_INPUT.rglob("train"):
    if p.is_dir() and (p / "images").exists():
        DATASET_PATH = p.parent
        break

if not DATASET_PATH:
    raise FileNotFoundError("Không tìm thấy thư mục dataset hợp lệ!")
print(f"✅ Đã tìm thấy dataset: {DATASET_PATH}")

# 3. Tạo ảnh low-light tổng hợp cho tập train (50% ảnh)
augment_dataset_low_light(DATASET_PATH / "train" / "images", ratio=0.5)

In [ ]:
# 3. Tạo file YAML tự động
yaml_content = f"""
path: {DATASET_PATH}
train: train/images
val: val/images
nc: 7
names: ['pedestrian', 'bicycle', 'motorbike', 'bus', 'truck', 'container truck', 'car']
"""
YAML_FILE = WORKING_DIR / "dataset.yaml"
YAML_FILE.write_text(yaml_content.strip())

# 4. Load Model
model = YOLO(str(MODEL_PATH))

In [ ]:
# ================================================================
# TRAINING TỐI ƯU THEO 3L-YOLO CHO ĐIỀU KIỆN THIẾU SÁNG
# Thay đổi: SGD thay AdamW, lr=0.01, batch=24, epochs=200,
#           multi_scale=True, close_mosaic=15
# ================================================================
model.train(
    data=str(YAML_FILE),
    device=[0,1],

    # === EPOCHS & PATIENCE (3L-YOLO: 200 epochs) ===
    epochs=200,
    patience=50,

    # === IMAGE SIZE ===
    imgsz=640,
    multi_scale=True,        # Giúp khái quát hóa tốt hơn cho phương tiện ở khoảng cách khác nhau
    close_mosaic=15,         # Tắt mosaic 15 epoch cuối để fine-tune chính xác hơn

    # === BATCH & OPTIMIZER (theo 3L-YOLO) ===
    batch=24,                # 3L-YOLO dùng 24 (tăng từ 8)
    optimizer="SGD",         # 3L-YOLO dùng SGD (thay AdamW)
    lr0=0.01,                # 3L-YOLO dùng 0.01 (tăng từ 0.001)
    lrf=0.01,
    momentum=0.937,          # 3L-YOLO dùng 0.937
    weight_decay=0.0005,     # 3L-YOLO dùng 5e-4
    warmup_epochs=5,
    cos_lr=True,

    # === AUGMENTATION TĂNG CƯỜNG CHO LOW-LIGHT ===
    mosaic=1.0,
    mixup=0.2,
    copy_paste=0.1,
    scale=0.6,
    translate=0.1,
    degrees=10.0,
    fliplr=0.5,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,               # Đa dạng hóa brightness mô phỏng điều kiện ánh sáng

    # === PERFORMANCE ===
    cache=True,
    workers=8,

    name="3l_yolo_traffic_lowlight"
)

In [ ]:
# ================================================================
# PREDICTION TỐI ƯU CHO ĐIỀU KIỆN THIẾU SÁNG
# - Giảm conf threshold xuống 0.20 (ưu tiên recall cho an toàn GT)
# - Bật Test-Time Augmentation (augment=True)
# ================================================================
sample_images = list((DATASET_PATH / "val" / "images").glob("*.jpg"))

if len(sample_images) == 0:
    print("Không có ảnh trong val/images")
else:
    best_weights = WORKING_DIR / "runs" / "detect" / "3l_yolo_traffic_lowlight" / "weights" / "best.pt"

    if best_weights.exists():
        eval_model = YOLO(str(best_weights))
        print(f"Đã load best model: {best_weights}")
    else:
        eval_model = model
        print("Không tìm thấy file best.pt, sử dụng model hiện tại.")

    # Predict với cấu hình tối ưu cho low-light
    pred = eval_model.predict(
        source=str(sample_images[0]),
        conf=0.20,              # Hạ threshold cho low-light (thay vì 0.25)
        iou=0.5,
        augment=True,           # Test-Time Augmentation cải thiện recall
        max_det=100,
        save=True,
        device=[0]
    )
    print("Done predict.")

In [ ]:
# ================================================================
# ĐÁNH GIÁ RIÊNG TRÊN TẬP VAL (xem mAP chi tiết từng class)
# ================================================================
if best_weights.exists():
    eval_model = YOLO(str(best_weights))
else:
    eval_model = model

metrics = eval_model.val(
    data=str(YAML_FILE),
    conf=0.20,
    iou=0.5,
    device=[0]
)
print(f"mAP@0.5: {metrics.box.map50:.4f}")
print(f"mAP@0.5:0.95: {metrics.box.map:.4f}")